# Solution 5: Feature Extraction by Country - Extremes DT

Extract high-resolution weather data for your country from the Extremes DT. This solution uses Spain as an example.

## Authentication

In [ ]:
%%capture cap
%run ../../desp-authentication.py

In [ ]:
output_1 = cap.stdout.split('}\n')
access_token = output_1[-1][0:-1]

## Imports

In [ ]:
import earthkit.data
import earthkit.plots
import earthkit.geo.cartography
import os

## Live Request Toggle

In [ ]:
LIVE_REQUEST = os.getenv("LIVE_REQUEST", "true").lower() == "true"
LIVE_REQUEST

## Select Your Country

In [ ]:
countries = ["Spain"]

shapes = earthkit.geo.cartography.country_polygons(countries, resolution=50e6)
print(f"Extracted {len(shapes)} polygon(s) for {countries}")

## Build the Request

The Extremes DT feature extraction request uses relative dates and includes a `feature` block.

In [ ]:
request = {
    "class": "d1",
    "dataset": "extremes-dt",
    "expver": "0001",
    "stream": "oper",
    "date": "-14",
    "time": "0000",
    "type": "fc",
    "levtype": "sfc",
    "step": "0",
    "param": "167",
    "feature": {
        "type": "polygon",
        "shape": shapes,
    },
}

## Retrieve the Data

In [ ]:
if LIVE_REQUEST:
    data = earthkit.data.from_source(
        "polytope", "destination-earth", request,
        address="polytope.lumi.apps.dte.destination-earth.eu",
        stream=False,
    )
else:
    data = earthkit.data.from_source("file", "../../extremes-dt/data/extremes-dt-fe-country-training.grib")

## Convert to xarray

In [ ]:
ds = data.to_xarray()
ds

## Plot Your Country

In [ ]:
chart = earthkit.plots.Map(domain=["Spain"])

chart.point_cloud(ds["2t"], units="celsius")
chart.coastlines()
chart.borders()
chart.gridlines()
chart.title("Extremes DT - 2m Temperature for Spain")
chart.legend()
chart.show()

## Bonus: Compare Two Countries

In [ ]:
# Extract data for both Spain and Portugal
countries_both = ["Spain", "Portugal"]
shapes_both = earthkit.geo.cartography.country_polygons(countries_both, resolution=50e6)

request_both = request.copy()
request_both["feature"] = {
    "type": "polygon",
    "shape": shapes_both,
}

if LIVE_REQUEST:
    data_both = earthkit.data.from_source(
        "polytope", "destination-earth", request_both,
        address="polytope.lumi.apps.dte.destination-earth.eu",
        stream=False,
    )
    ds_both = data_both.to_xarray()

    chart = earthkit.plots.Map(domain=["Spain", "Portugal"])
    chart.point_cloud(ds_both["2t"], units="celsius")
    chart.coastlines()
    chart.borders()
    chart.gridlines()
    chart.title("Extremes DT - 2m Temperature for Spain & Portugal")
    chart.legend()
    chart.show()